# Energy Economics Curated Dataset - EDA

Exploratory Data Analysis of a curated dataset covering 129 countries from 1995 to 2020, combining energy consumption, economic indicators, climate data, and environmental metrics.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12

## 1. Data Loading & Cleaning

In [ ]:
df = pd.read_csv('/kaggle/input/datasets/sandhyapalaniappan/energy-economics-curated-dataset/energy_economics_curated.csv')

# Convert string columns to numeric
for col in ['foreign_direct_investment_net_inflows_pct_gdp', 'real_gdp_constant_2015_usd', 'natural_capital_dependency_index']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

print(f'Shape: {df.shape}')
print(f'Countries: {df["country_name"].nunique()}')
print(f'Year range: {df["observation_year"].min()} - {df["observation_year"].max()}')
df.head()

In [ ]:
df.info()

In [ ]:
# Missing values
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(1)
missing_df = pd.DataFrame({'count': missing, 'pct': missing_pct}).query('count > 0').sort_values('pct', ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(missing_df.index, missing_df['pct'], color='coral')
ax.set_xlabel('Missing %')
ax.set_title('Missing Values by Feature')
for i, (v, c) in enumerate(zip(missing_df['pct'], missing_df['count'])):
    ax.text(v + 0.3, i, f'{v}% ({c})', va='center', fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
df.describe().round(3)

## 2. Global Trends Over Time

In [ ]:
yearly_avg = df.groupby('observation_year').agg(
    co2_per_capita=('co2_emissions_metric_tonnes_per_capita', 'mean'),
    renewable_pct=('renewable_energy_consumption_pct_final_energy_use', 'mean'),
    gdp=('real_gdp_constant_2015_usd', 'mean'),
    urban_pct=('urban_population_pct_total_population', 'mean'),
    ghg_total=('total_greenhouse_gas_emissions_kt_co2e', 'sum'),
    avg_temp=('average_temperature_celsius', 'mean')
).reset_index()

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

axes[0, 0].plot(yearly_avg['observation_year'], yearly_avg['co2_per_capita'], marker='o', markersize=4, color='indianred')
axes[0, 0].set_title('Avg CO2 Emissions per Capita')
axes[0, 0].set_ylabel('Metric Tonnes')

axes[0, 1].plot(yearly_avg['observation_year'], yearly_avg['renewable_pct'], marker='o', markersize=4, color='green')
axes[0, 1].set_title('Avg Renewable Energy Consumption %')
axes[0, 1].set_ylabel('%')

axes[0, 2].plot(yearly_avg['observation_year'], yearly_avg['gdp'] / 1e9, marker='o', markersize=4, color='steelblue')
axes[0, 2].set_title('Avg Real GDP (constant 2015 USD)')
axes[0, 2].set_ylabel('Billion USD')

axes[1, 0].plot(yearly_avg['observation_year'], yearly_avg['urban_pct'], marker='o', markersize=4, color='purple')
axes[1, 0].set_title('Avg Urban Population %')
axes[1, 0].set_ylabel('%')

axes[1, 1].plot(yearly_avg['observation_year'], yearly_avg['ghg_total'] / 1e6, marker='o', markersize=4, color='darkorange')
axes[1, 1].set_title('Total GHG Emissions (all countries)')
axes[1, 1].set_ylabel('Million kt CO2e')

axes[1, 2].plot(yearly_avg['observation_year'], yearly_avg['avg_temp'], marker='o', markersize=4, color='red')
axes[1, 2].set_title('Avg Temperature')
axes[1, 2].set_ylabel('Celsius')

for ax in axes.flat:
    ax.set_xlabel('Year')

plt.suptitle('Global Trends (1995-2020)', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

## 3. Top Emitters & Renewable Leaders

In [ ]:
latest = df[df['observation_year'] == 2020].copy()

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# Top 20 CO2 per capita
top_co2 = latest.nlargest(20, 'co2_emissions_metric_tonnes_per_capita')
axes[0].barh(top_co2['country_name'], top_co2['co2_emissions_metric_tonnes_per_capita'], color='indianred')
axes[0].set_title('Top 20 CO2 Emitters per Capita (2020)')
axes[0].set_xlabel('Metric Tonnes per Capita')
axes[0].invert_yaxis()

# Top 20 renewable energy consumers
top_renew = latest.nlargest(20, 'renewable_energy_consumption_pct_final_energy_use')
axes[1].barh(top_renew['country_name'], top_renew['renewable_energy_consumption_pct_final_energy_use'], color='green')
axes[1].set_title('Top 20 Renewable Energy Consumers (2020)')
axes[1].set_xlabel('% of Final Energy Use')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

In [ ]:
# Top 15 total GHG emitters
top_ghg = latest.nlargest(15, 'total_greenhouse_gas_emissions_kt_co2e')

fig, ax = plt.subplots(figsize=(12, 6))
ax.barh(top_ghg['country_name'], top_ghg['total_greenhouse_gas_emissions_kt_co2e'] / 1e6, color='darkorange')
ax.set_title('Top 15 Total GHG Emitters (2020)')
ax.set_xlabel('Million kt CO2e')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 4. Correlation Analysis

In [ ]:
corr_cols = [
    'adaptive_capacity_index', 'foreign_direct_investment_net_inflows_pct_gdp',
    'forest_area_pct_land_area', 'real_gdp_constant_2015_usd',
    'renewable_energy_consumption_pct_final_energy_use',
    'urban_population_pct_total_population', 'readiness_index',
    'ecological_footprint_index', 'co2_emissions_metric_tonnes_per_capita',
    'total_greenhouse_gas_emissions_kt_co2e', 'average_temperature_celsius',
    'total_population', 'greenhouse_gas_emissions_metric_tonnes_per_capita'
]

# Shorter labels for readability
short_labels = [
    'Adaptive Cap.', 'FDI % GDP', 'Forest %', 'Real GDP',
    'Renewable %', 'Urban %', 'Readiness', 'Eco Footprint',
    'CO2/capita', 'Total GHG', 'Avg Temp', 'Population', 'GHG/capita'
]

corr = df[corr_cols].corr()
corr.index = short_labels
corr.columns = short_labels

fig, ax = plt.subplots(figsize=(14, 12))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            square=True, linewidths=0.5, ax=ax)
ax.set_title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

## 5. GDP vs Emissions Relationship

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# GDP per capita vs CO2 per capita (2020)
latest['gdp_per_capita'] = latest['real_gdp_constant_2015_usd'] / latest['total_population']

axes[0].scatter(latest['gdp_per_capita'], latest['co2_emissions_metric_tonnes_per_capita'],
                s=latest['total_population'] / 1e6, alpha=0.5, edgecolors='black', linewidth=0.5)
axes[0].set_xlabel('GDP per Capita (USD, 2015 constant)')
axes[0].set_ylabel('CO2 Emissions per Capita (t)')
axes[0].set_title('GDP per Capita vs CO2 per Capita (2020)\nBubble size = population')
axes[0].set_xscale('log')

# Annotate some notable countries
for _, row in latest.nlargest(5, 'co2_emissions_metric_tonnes_per_capita').iterrows():
    axes[0].annotate(row['country_name'], (row['gdp_per_capita'], row['co2_emissions_metric_tonnes_per_capita']),
                     fontsize=8, ha='left')

# Renewable % vs CO2 per capita
axes[1].scatter(latest['renewable_energy_consumption_pct_final_energy_use'],
                latest['co2_emissions_metric_tonnes_per_capita'],
                s=latest['total_population'] / 1e6, alpha=0.5, edgecolors='black', linewidth=0.5, color='green')
axes[1].set_xlabel('Renewable Energy Consumption (%)')
axes[1].set_ylabel('CO2 Emissions per Capita (t)')
axes[1].set_title('Renewable Energy % vs CO2 per Capita (2020)\nBubble size = population')

plt.tight_layout()
plt.show()

## 6. Regional & Climate Patterns

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Temperature vs CO2 per capita
axes[0].scatter(df['average_temperature_celsius'], df['co2_emissions_metric_tonnes_per_capita'],
                alpha=0.1, s=5, color='indianred')
axes[0].set_xlabel('Average Temperature (Celsius)')
axes[0].set_ylabel('CO2 per Capita (t)')
axes[0].set_title('Temperature vs CO2 Emissions per Capita')

# Forest area vs ecological footprint
axes[1].scatter(df['forest_area_pct_land_area'], df['ecological_footprint_index'],
                alpha=0.1, s=5, color='forestgreen')
axes[1].set_xlabel('Forest Area (% of Land)')
axes[1].set_ylabel('Ecological Footprint Index')
axes[1].set_title('Forest Coverage vs Ecological Footprint')

plt.tight_layout()
plt.show()

In [ ]:
# Urbanization vs Renewable Energy over time (selected countries)
highlight = ['United States', 'China', 'India', 'Germany', 'Brazil', 'Japan', 'Nigeria', 'Australia']
subset = df[df['country_name'].isin(highlight)]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for country in highlight:
    c = subset[subset['country_name'] == country]
    axes[0].plot(c['observation_year'], c['co2_emissions_metric_tonnes_per_capita'], label=country, marker='o', markersize=3)
    axes[1].plot(c['observation_year'], c['renewable_energy_consumption_pct_final_energy_use'], label=country, marker='o', markersize=3)

axes[0].set_title('CO2 per Capita Over Time')
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Metric Tonnes per Capita')
axes[0].legend(fontsize=8)

axes[1].set_title('Renewable Energy % Over Time')
axes[1].set_xlabel('Year')
axes[1].set_ylabel('% of Final Energy Use')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

## 7. Readiness & Adaptive Capacity

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Readiness vs Adaptive Capacity (2020)
axes[0].scatter(latest['readiness_index'], latest['adaptive_capacity_index'],
                s=latest['total_population'] / 1e6, alpha=0.5, edgecolors='black', linewidth=0.5, color='teal')
axes[0].set_xlabel('Readiness Index')
axes[0].set_ylabel('Adaptive Capacity Index')
axes[0].set_title('Readiness vs Adaptive Capacity (2020)\nBubble size = population')

# Annotate extremes
for _, row in latest.nlargest(3, 'readiness_index').iterrows():
    axes[0].annotate(row['country_name'], (row['readiness_index'], row['adaptive_capacity_index']),
                     fontsize=8)
for _, row in latest.nsmallest(3, 'readiness_index').dropna(subset=['readiness_index']).iterrows():
    axes[0].annotate(row['country_name'], (row['readiness_index'], row['adaptive_capacity_index']),
                     fontsize=8)

# Readiness index distribution
axes[1].hist(latest['readiness_index'].dropna(), bins=25, color='teal', alpha=0.7, edgecolor='white')
axes[1].axvline(latest['readiness_index'].median(), color='red', linestyle='--',
                label=f'Median: {latest["readiness_index"].median():.3f}')
axes[1].set_title('Distribution of Readiness Index (2020)')
axes[1].set_xlabel('Readiness Index')
axes[1].legend()

plt.tight_layout()
plt.show()

## 8. Distribution of Key Features

In [ ]:
features = [
    ('co2_emissions_metric_tonnes_per_capita', 'CO2 per Capita'),
    ('renewable_energy_consumption_pct_final_energy_use', 'Renewable Energy %'),
    ('urban_population_pct_total_population', 'Urban Population %'),
    ('forest_area_pct_land_area', 'Forest Area %'),
    ('ecological_footprint_index', 'Ecological Footprint'),
    ('average_temperature_celsius', 'Avg Temperature')
]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax, (col, title) in zip(axes.flat, features):
    ax.hist(df[col].dropna(), bins=40, alpha=0.7, edgecolor='white', color='steelblue')
    ax.set_title(title)
    ax.axvline(df[col].mean(), color='red', linestyle='--', alpha=0.7)

plt.suptitle('Feature Distributions (all years)', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 9. Change Over Time (1995 vs 2020)

In [ ]:
start = df[df['observation_year'] == 1995][['country_name', 'co2_emissions_metric_tonnes_per_capita',
                                              'renewable_energy_consumption_pct_final_energy_use',
                                              'urban_population_pct_total_population']].set_index('country_name')
end = df[df['observation_year'] == 2020][['country_name', 'co2_emissions_metric_tonnes_per_capita',
                                            'renewable_energy_consumption_pct_final_energy_use',
                                            'urban_population_pct_total_population']].set_index('country_name')

change = (end - start).dropna()
change.columns = ['CO2/capita change', 'Renewable % change', 'Urban % change']

fig, axes = plt.subplots(1, 3, figsize=(18, 7))

# CO2 change
top_inc = change.nlargest(10, 'CO2/capita change')
top_dec = change.nsmallest(10, 'CO2/capita change')
combined = pd.concat([top_inc, top_dec]).sort_values('CO2/capita change')
colors = ['green' if x < 0 else 'red' for x in combined['CO2/capita change']]
axes[0].barh(combined.index, combined['CO2/capita change'], color=colors)
axes[0].set_title('CO2 per Capita Change (1995-2020)')
axes[0].set_xlabel('Change (t)')

# Renewable change
top_inc_r = change.nlargest(10, 'Renewable % change')
top_dec_r = change.nsmallest(10, 'Renewable % change')
combined_r = pd.concat([top_inc_r, top_dec_r]).sort_values('Renewable % change')
colors_r = ['red' if x < 0 else 'green' for x in combined_r['Renewable % change']]
axes[1].barh(combined_r.index, combined_r['Renewable % change'], color=colors_r)
axes[1].set_title('Renewable Energy % Change (1995-2020)')
axes[1].set_xlabel('Change (pp)')

# Urban change
top_urban = change.nlargest(15, 'Urban % change')
axes[2].barh(top_urban.index, top_urban['Urban % change'], color='purple', alpha=0.7)
axes[2].set_title('Top 15 Urbanization Growth (1995-2020)')
axes[2].set_xlabel('Change (pp)')
axes[2].invert_yaxis()

plt.tight_layout()
plt.show()

## 10. Key Findings

Summary of the exploratory analysis:

- **Rising emissions**: Total global GHG emissions have trended upward from 1995-2020 despite some countries reducing per-capita emissions
- **Renewable energy**: Average renewable energy consumption share has declined slightly globally, though some countries (e.g., Germany) have made significant gains
- **GDP-Emissions link**: Higher GDP per capita generally correlates with higher CO2 per capita, but the relationship is not linear - some wealthy nations have decoupled growth from emissions
- **Temperature & emissions**: Colder countries tend to have higher per-capita CO2 emissions, likely driven by heating demand and industrial economies
- **Readiness gap**: Large disparity in climate readiness and adaptive capacity between developed and developing nations
- **Urbanization trend**: Steady increase in urban population globally, with fastest growth in African and Asian countries
- **Missing data**: Official development assistance has the most missing values (20.6%), followed by natural capital dependency index (8.5%)